류현수 2026-10-01 10:26

# Part 1. 객관식

1. ②
2. ③
3. ③
4. ①
5. ②
6. ③
7. ②
8. ②
9. ②
10. ③

# Part 2. 주관식

## 1
토큰은 LLM에서 사용량을 계산하는 단위이며 LLM 모델이 문장을 처리하는 단위입니다.
주로 영어는 단어 단위와 비슷하지만, 한글은 단어보다 더 짧은 단위로 나누어지기 때문에 보통 영어가 토큰 수가 더 작게 계산됩니다.

## 2
`max_tokens`를 작게 수정하여 출력 토큰 수를 제한하는 방법이 있다. 입력 토큰보다 출력 토큰이 비용에 4배 더 큰 영향을 미치기 때문에, 출력 토큰을 제한하는 것이 예산 관리에 더 효율적이다.

## 3
`temperature` 파라미터를 0으로 설정하고, `max_tokens`를 10 정도로 낮게 설정해야한다.
`temperature`를 낮추면 더 일관적이고 논리적인 응답이 생성된다.
또한 필요한 출력의 길이가 짧기 때문에 `max_tokens`를 제한하여 필요한 응답만 생성되게 유도할 수 있다.

## 4
후보 B가 더 적합하다.
후보 A에는 LLM의 환각을 방지하기 위한 수단이 전혀 없으며,
형식 예제 및 길이 제한이 주어지지 않아 일관된 응답 형식을 기대할 수 없다.

## 5  

잘 작동하는 페르소나 system 메시지에는 다음 3가지 요소가 필요하다.

- 역할과 행동을 명확하게 정의한다.
- 형식과 길이를 정의한다.
- 환각을 방지하도록 유도한다.

## 6

In [3]:
import os

from dotenv import load_dotenv
load_dotenv()  # .env 에서 환경 변수를 로드

api_key = os.environ.get("OPENAI_API_KEY", "") # OPENAI_API_KEY 환경 변수를 api_key에 저장
print(f"API 키 로드: {api_key[:7]}..." if api_key else "❌ 키를 못 찾았어요")

✅ 로컬 Jupyter 환경입니다.
API 키 로드: sk-proj...


In [4]:
from openai import OpenAI

client = OpenAI() # OpenAI 클라이언트 객체를 생성
print("✅ OpenAI 클라이언트 준비 완료")

# temperature:0.3, max_tokens:500을 기본 값으로 가진 llm 요청 함수 정의
def ask_llm(messages, temperature=0.3, max_tokens=500):
    response = client.chat.completions.create(
        model="gpt-4o-mini", # gpt-4o-mini 모델을 사용
        # create 메서드에 ask_llm 함수 인자값 전달
        messages=messages, 
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content # LLM 출력 메세지를 추출하여 반환

✅ OpenAI 클라이언트 준비 완료


In [5]:
print(ask_llm([
    {"role":"user", "content": "RAG가 무엇인가요?"}
])) # ask_llm을 통해 user role로 질문 전달

RAG는 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 생성 모델을 결합한 자연어 처리(NLP) 기술입니다. 이 접근 방식은 주어진 질문이나 요청에 대해 더 정확하고 풍부한 답변을 제공하기 위해 외부 데이터베이스나 문서에서 정보를 검색한 후, 이를 바탕으로 텍스트를 생성하는 방식입니다.

RAG의 주요 구성 요소는 다음과 같습니다:

1. **정보 검색 (Retrieval)**: 사용자의 질문에 관련된 정보를 외부 데이터 소스(예: 문서, 웹 페이지 등)에서 검색합니다. 이 과정은 일반적으로 검색 엔진이나 벡터 검색 기술을 사용합니다.

2. **텍스트 생성 (Generation)**: 검색된 정보를 바탕으로 자연어로 답변을 생성합니다. 이 단계에서는 주로 언어 모델이 사용됩니다.

RAG는 특히 정보가 풍부하고 다양한 배경 지식이 필요한 질문에 대해 유용하며, 최신 정보에 접근할 수 있는 장점이 있습니다. 예를 들어, 특정 주제에 대한 최신 연구 결과나 통계 데이터를 포함한 답변을 제공할 수 있습니다.


## 7

In [20]:
messages = [
    # system role 사용, 한국어 비서 역할임을 지정
    {"role": "system", "content": "당신은 친절한 한국어 비서입니다."},
    # user role 사용, 대화 내용임을 의미
    {"role": "user", "content": "제 이름은 현수입니다."},
    # assistant role 사용, 과거 응답임을 의미
    {"role": "assistant", "content": "안녕하세요, 현수님! 무엇을 도와드릴까요?"},
    # user role 사용, 마지막 user role 이므로 이 content에 맞는 응답 생성
    {"role": "user", "content": "제 이름이 뭐였죠?"}
]

print(ask_llm(messages))

현수님이라고 하셨습니다! 다른 질문이나 도움이 필요하신 부분이 있으면 말씀해 주세요.


## 8

In [16]:
def classify_inquiry(text: str) -> str:
    messages = [
        {"role": "system", "content": "결제 / 배송 / 환불 / 기타 중 하나로 분류"},
        {"role": "user", "content": text},
    ]

    return ask_llm(messages, temperature=0, max_tokens=10)

test_cases = [
"어제 주문했는데 카드 결제가 두 번 되었어요.",
"택배가 일주일째 안 와요.",
"사이즈가 안 맞아서 반품하고 싶어요.",
"영업시간이 어떻게 되나요?",
]


for case in test_cases:
    category = classify_inquiry(case)
    print(f"[{category:>4}] {case}")


[  결제] 어제 주문했는데 카드 결제가 두 번 되었어요.
[  배송] 택배가 일주일째 안 와요.
[반품은 '환불'로 분류] 사이즈가 안 맞아서 반품하고 싶어요.
[  기타] 영업시간이 어떻게 되나요?


## 9

In [18]:
prompt = "AI 스타트업 이름 3개를 지어주세요."

for temp in [0.0, 0.0, 1.2, 1.2]:
    answer = ask_llm(
        [{"role": "user", "content": prompt}],
        temperature=temp,
    )
    print(f"🌡 temperature={temp}\n→ {answer}\n")

🌡 temperature=0.0
→ 물론입니다! 다음은 AI 스타트업을 위한 이름 세 가지입니다:

1. **인사이트AI (InsightAI)** - 데이터 분석과 인사이트 제공에 중점을 둔 스타트업에 적합한 이름입니다.
  
2. **넥스트젠AI (NextGenAI)** - 차세대 기술과 혁신을 강조하는 이름으로, 미래 지향적인 이미지를 줍니다.

3. **스마트코어 (SmartCore)** - AI의 핵심 기술을 다루는 스타트업에 어울리는 이름으로, 지능적이고 효율적인 솔루션을 암시합니다.

이 이름들이 도움이 되길 바랍니다!

🌡 temperature=0.0
→ 물론입니다! 다음은 AI 스타트업을 위한 이름 세 가지입니다:

1. **인사이트AI (InsightAI)** - 데이터 분석과 인사이트 제공에 중점을 둔 스타트업에 적합한 이름입니다.
  
2. **넥스트젠AI (NextGenAI)** - 차세대 기술과 혁신을 강조하는 이름으로, 미래 지향적인 AI 솔루션을 제공하는 스타트업에 어울립니다.

3. **스마트코어 (SmartCore)** - AI의 핵심 기술을 바탕으로 다양한 산업에 적용할 수 있는 솔루션을 개발하는 스타트업에 적합한 이름입니다.

이 이름들이 도움이 되길 바랍니다!

🌡 temperature=1.2
→ 물론입니다! 다음은 AI 스타트업을 위한 세 가지 이름 제안입니다:

1. **인사이트 AI (Insight AI)**  
   - 데이터 분석과 통찰력을 제공하는 AI 솔루션을 연상시킵니다.

2. **뮤즈널리틱스 (Muse Analytics)**  
   - 창의성과 분석을 결합한 느낌을 주며, 예술과 데이터의 융합을 뜻합니다.

3. **퓨처퓨전 (FutureFusion)**  
   - 미래 지향적인 AI 기술들을 융합하여 새로운 가능성을 창출하는 이미지를 강조합니다.

원하는 스타일이나 방향성이 있다면 추가적으로 맞춤형 아이디어를 제공해드릴 수 있습니다!

🌡 temperature=1.2
→ 물론입니다! 아래는 AI 스타트

`temperature`가 0일 때는 인사이트AI, 넥스트젠AI, 스마트코어 3가지로 일관되게 나왔지만,
1.2일 때는 각 결과가 달랐다.

# 10

In [19]:
PRICE_INPUT = 0.15 # 입력 1M 토큰당 USD
PRICE_OUTPUT = 0.60 # 출력 1M 토큰당 USD
USD_TO_KRW = 1400 # 환율

def cost_korean_won(usage):
    """usage 객체를 받아 비용을 원화로 환산"""
    input_cost = usage.prompt_tokens / 1_000_000 * PRICE_INPUT
    output_cost = usage.completion_tokens / 1_000_000 * PRICE_OUTPUT
    return (input_cost + output_cost) * USD_TO_KRW

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Python의 장점을 5가지 설명해주세요."}],
)

u = response.usage
print(f"입력 {u.prompt_tokens} + 출력 {u.completion_tokens} = "
          f"총 {u.total_tokens} 토큰, 비용 약 {cost_korean_won(u):.4f}원")

입력 17 + 출력 478 = 총 495 토큰, 비용 약 0.4051원


# 체크리스트

[x] 파일명이 Day1_Lesson1_Test_본인이름.ipynb 형식인가
[x] 노트북 첫 셀에 성명과 제출일시를 기재했는가
[x] 객관식 10문항의 답을 한 셀에 번호 순서대로 정리했는가
[x] 주관식 6~10번 코드 셀을 모두 실행해 출력 결과가 남아 있는가
[x] API 키가 코드에 그대로 노출되어 있지 않은가
[x] 저장한 파일을 다시 열어 정상 표시되는지 확인했는가
[] 09:50 전에 과제함에 제출했는가